# Chapter 2: Nearby Friends
*System Design Interview Volume 2*

## TL;DR

The "Nearby Friends" feature shows which of your friends are geographically close in real time. Unlike the static proximity service, this requires **dynamic, real-time location updates**. The architecture uses **WebSocket** for bi-directional communication and **Redis Pub/Sub** as a lightweight routing layer where each user has a dedicated channel. At 10M concurrent users updating every 30 seconds, the system must handle 334K location updates/sec and fan out ~14M updates/sec to friends.

## Requirements

| Requirement | Detail |
|---|---|
| Nearby friend list | Show friends within 5 miles with distance + timestamp |
| Real-time updates | Refresh every few seconds |
| Low latency | Friends see location changes without noticeable delay |
| Reliability | Occasional data point loss is acceptable |
| Eventual consistency | A few seconds delay across replicas is fine |
| Inactivity timeout | Friends inactive >10 min disappear from list |

## Estimation: Location Update Scale

In [ ]:
# Back-of-the-envelope: Nearby Friends
dau = 100_000_000           # 100M DAU
concurrent_pct = 0.10       # 10% concurrent
concurrent_users = int(dau * concurrent_pct)
refresh_interval = 30       # seconds

location_update_qps = concurrent_users / refresh_interval
print(f"Concurrent users: {concurrent_users:,}")
print(f"Location update QPS: {location_update_qps:,.0f}")

# Fan-out calculation
avg_friends = 400
online_nearby_pct = 0.10    # 10% of friends online & nearby
fan_out_per_sec = location_update_qps * avg_friends * online_nearby_pct
print(f"Fan-out updates/sec: {fan_out_per_sec:,.0f}")

# Redis Pub/Sub server estimate
channels = dau * 0.10       # 10% use feature = 100M channels
active_subscribers_per_channel = 100  # avg active friends per channel
bytes_per_subscriber = 20   # hash table + linked list pointer
memory_gb = channels * bytes_per_subscriber * active_subscribers_per_channel / (1024**3)
print(f"\nRedis Pub/Sub memory: {memory_gb:.0f} GB")

pushes_per_server = 100_000  # conservative estimate per server
servers_needed = fan_out_per_sec / pushes_per_server
print(f"Redis Pub/Sub servers needed (CPU-bound): {servers_needed:.0f}")

## High-Level Design

```mermaid
graph TB
    Users[Mobile Users]
    LB[Load Balancer]
    WS[WebSocket Servers<br>stateful, bi-directional]
    API[REST API Servers<br>stateless]
    PS[Redis Pub/Sub<br>per-user channels]
    LC[Redis Location Cache<br>TTL-based]
    LH[(Location History DB<br>Cassandra)]
    UDB[(User DB<br>profiles + friendships)]

    Users -->|WebSocket| LB
    Users -->|HTTP| LB
    LB --> WS
    LB --> API
    WS -->|publish location| PS
    WS -->|subscribe friends| PS
    WS --> LC
    WS --> LH
    API --> UDB

    style WS fill:#b3e6b3,stroke:#333
    style PS fill:#ff9999,stroke:#333
    style LC fill:#ffcc99,stroke:#333
```

### Periodic Location Update Flow

1. Client sends location update via **WebSocket** to load balancer
2. Load balancer forwards to the client's **WebSocket server**
3. Server writes to **location history DB** (Cassandra)
4. Server updates **location cache** (Redis, refreshes TTL)
5. Server **publishes** to user's channel in Redis Pub/Sub
6. Pub/Sub **broadcasts** to all friend subscribers
7. Each subscriber's WebSocket handler **computes distance** and forwards if within radius

## Deep Dive: Redis Pub/Sub Routing

Each user gets a dedicated Pub/Sub channel. When User 1 updates location:

```mermaid
sequenceDiagram
    participant U1 as User 1 Client
    participant WS1 as WS Server A
    participant PS as Redis Pub/Sub
    participant WS2 as WS Server B
    participant U2 as User 2 Client

    U1->>WS1: Location update (lat, lng, ts)
    WS1->>PS: PUBLISH user_1_channel
    PS->>WS2: Broadcast to subscriber (User 2 handler)
    WS2->>WS2: Compute distance(User1, User2)
    alt distance < radius
        WS2->>U2: Forward location + timestamp
    else distance >= radius
        WS2->>WS2: Drop update
    end
```

**Key properties of Redis Pub/Sub:**
- Channels are extremely cheap to create (~20 bytes per subscriber pointer)
- Messages are fire-and-forget (not persisted)
- CPU is the bottleneck, not memory (~200 GB memory vs ~140 servers for CPU)

## Deep Dive: Distributed Pub/Sub Cluster

```mermaid
graph LR
    SD[Service Discovery<br>etcd / ZooKeeper]
    SD -->|hash ring config| WS1[WS Server 1]
    SD -->|hash ring config| WS2[WS Server 2]
    WS1 -->|publish| PS1[Pub/Sub Server 1]
    WS1 -->|subscribe| PS2[Pub/Sub Server 2]
    WS2 -->|publish| PS2
    WS2 -->|subscribe| PS1

    subgraph HR["Hash Ring (consistent hashing)"]
        PS1
        PS2
        PS3[Pub/Sub Server N]
    end

    style SD fill:#e6ccff,stroke:#333
```

- Channels are sharded across Pub/Sub servers using a **consistent hash ring**
- Hash ring stored in **service discovery** (etcd/ZooKeeper)
- WebSocket servers cache the ring locally and subscribe to updates
- **Resizing triggers mass resubscriptions** -- treat cluster as stateful, over-provision
- Replacing a single dead server only moves channels on that server

## Deep Dive: Client Initialization

When a user opens the app:

1. Establish WebSocket connection
2. Update location in Redis cache + save to connection handler variable
3. Load all friends from User DB
4. Batch-fetch friend locations from Redis cache (absent = inactive, TTL expired)
5. Compute distances, return nearby friends to client
6. Subscribe to **all** friend channels (active + inactive) in Redis Pub/Sub
7. Publish own location to own channel

## Trade-offs

| Decision | Pro | Con |
|---|---|---|
| WebSocket over HTTP polling | True real-time, lower overhead | Stateful servers, complex scaling |
| Redis Pub/Sub over Kafka | Lightweight channels, low latency | No persistence, messages lost if no subscriber |
| Per-user channels | Simple routing, no topic management | 100M+ channels, memory overhead |
| Subscribe to all friends (incl. inactive) | Simpler logic, no subscribe/unsubscribe on status change | Higher memory usage |
| Consistent hash ring for Pub/Sub | Even distribution, easy server replacement | Resizing causes resubscription storms |
| Erlang/BEAM alternative | Native concurrency, replaces Pub/Sub entirely | Niche expertise, hard to hire |

## Takeaways

1. **Real-time location** requires stateful WebSocket connections, not REST polling
2. **Redis Pub/Sub** is ideal for lightweight fan-out with millions of channels
3. **TTL on cache entries** elegantly handles user inactivity without explicit cleanup
4. **CPU, not memory** is the Pub/Sub bottleneck -- plan server count around throughput
5. **Service discovery + consistent hashing** enables scaling the Pub/Sub cluster
6. **Erlang/BEAM** is a powerful alternative where each user is a lightweight process (~300 bytes)

## Related Concepts

- [[websocket-real-time]] -- Persistent bi-directional communication for location
- [[redis-pub-sub]] -- Lightweight per-user channel routing
- [[distributed-pub-sub]] -- Scaling Pub/Sub with consistent hashing
- [[location-cache-ttl]] -- Redis cache with TTL for active user locations
- [[erlang-alternative]] -- BEAM VM as a Pub/Sub replacement